<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
# ============================================================
# ML-07 / W04 — BASELINE ACTION SCORE
# STEP 0: LOAD FINAL FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("W04 — BASELINE ACTION SCORE")
print("STEP 0: LOAD FINAL FEATURE DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Load the 20-column processed Parquet dataset
# ------------------------------------------------------------

df_features = pd.read_parquet(
    "/content/final_features_clean.parquet"
)

# Work on a copy
df_baseline = df_features.copy()

# ------------------------------------------------------------
# Basic inspection
# ------------------------------------------------------------

print("\nDataset loaded successfully.")

print(f"Rows    : {len(df_baseline):,}")
print(f"Columns : {len(df_baseline.columns)}")

print("\nColumns:")
for i, col in enumerate(df_baseline.columns, start=1):
    print(f"{i:2}. {col}")

print("\nShape:")
print(df_baseline.shape)

print("\nFirst 5 rows:")
display(df_baseline.head())

W04 — BASELINE ACTION SCORE
STEP 0: LOAD FINAL FEATURE DATASET

Dataset loaded successfully.
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape:
(2871202, 19)

First 5 rows:


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,missing_count,gsc_avg_position_missing,ga4_total_engagement_sec_missing,sessions_organic_missing,sessions_ai_missing,ai_other_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0,0,0,0,0,0.006849,0.0,0.0,0.0
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0


**SIGNAL TEST #1** : Flyrank Staleness Signal

In [10]:
# ============================================================
# FINAL STALENESS DECAY RATE TEST
# ============================================================
#
# Uses df_baseline
# Original dataframe remains untouched.
#
# Test:
#   90-180 days
#   181-365 days
#   365+ days
#
# Decay:
#   Future 3-month impressions decrease >= 20%
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. PREPARE COPY
# ============================================================

df_decay_test = df_baseline.copy()

df_decay_test["month"] = pd.to_datetime(
    df_decay_test["month"],
    errors="coerce"
)

df_decay_test = (
    df_decay_test
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. FIRST AND LAST OBSERVED MONTH
# ============================================================

first_month = (
    df_decay_test
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_decay_test
    .groupby("content_hash_id")["month"]
    .transform("max")
)


# ============================================================
# 3. CONTENT AGE
# ============================================================
#
# Age is the observed history of the page:
#
# last observed month - first observed month
#
# This matches the previous Staleness Test-1 logic.
# ============================================================

df_decay_test["content_age_days"] = (
    last_month - first_month
).dt.days


# ============================================================
# 4. 90-DAY ELIGIBILITY
# ============================================================

df_decay_test = df_decay_test[
    df_decay_test["content_age_days"] >= 90
].copy()


# ============================================================
# 5. CREATE EXACT AGE BUCKETS
# ============================================================

df_decay_test["age_bucket"] = np.select(
    [
        df_decay_test["content_age_days"] <= 180,

        df_decay_test["content_age_days"] <= 365,

        df_decay_test["content_age_days"] > 365
    ],
    [
        "90-180 days",
        "181-365 days",
        "365+ days"
    ],
    default="unknown"
)


# ============================================================
# 6. FUTURE 3-MONTH IMPRESSIONS
# ============================================================
#
# IMPORTANT:
# shift(-3) means the 3rd available observation.
#
# This preserves the same logic used in your original test.
# ============================================================

df_decay_test["future_impressions"] = (
    df_decay_test
    .groupby("content_hash_id")["gsc_impressions"]
    .shift(-3)
)


# ============================================================
# 7. FUTURE IMPRESSION CHANGE %
# ============================================================
#
# Parentheses are intentionally used around every condition.
# ============================================================

valid_future = (
    (df_decay_test["gsc_impressions"] > 0)
    &
    (df_decay_test["future_impressions"].notna())
)

df_decay_test["future_impression_change_pct"] = np.nan

df_decay_test.loc[
    valid_future,
    "future_impression_change_pct"
] = (
    (
        df_decay_test.loc[
            valid_future,
            "future_impressions"
        ]
        -
        df_decay_test.loc[
            valid_future,
            "gsc_impressions"
        ]
    )
    /
    df_decay_test.loc[
        valid_future,
        "gsc_impressions"
    ]
) * 100


# ============================================================
# 8. DEFINE DECAY
# ============================================================
#
# Decay = future impressions decreased by 20% or more.
# ============================================================

df_decay_test["decay"] = np.nan

valid_change = (
    df_decay_test[
        "future_impression_change_pct"
    ].notna()
)

df_decay_test.loc[
    valid_change,
    "decay"
] = (
    df_decay_test.loc[
        valid_change,
        "future_impression_change_pct"
    ] <= -20
)


# ============================================================
# 9. KEEP ONLY VALID FUTURE TARGET ROWS
# ============================================================

df_decay_test = df_decay_test[
    df_decay_test[
        "future_impression_change_pct"
    ].notna()
].copy()


# ============================================================
# 10. FINAL STALENESS DECAY TEST
# ============================================================

age_order = [
    "90-180 days",
    "181-365 days",
    "365+ days"
]

final_check = (
    df_decay_test
    .groupby(
        "age_bucket",
        observed=True
    )
    .agg(
        observations=(
            "decay",
            "size"
        ),

        decay_cases=(
            "decay",
            "sum"
        ),

        decay_rate=(
            "decay",
            "mean"
        ),

        avg_future_impression_change_pct=(
            "future_impression_change_pct",
            "mean"
        )
    )
    .reindex(age_order)
    .reset_index()
)


# ============================================================
# 11. PERCENTAGES
# ============================================================

final_check["decay_rate"] = (
    final_check["decay_rate"] * 100
).round(2)

final_check[
    "avg_future_impression_change_pct"
] = (
    final_check[
        "avg_future_impression_change_pct"
    ]
    .round(2)
)


# ============================================================
# 12. DISPLAY RESULT
# ============================================================

print("=" * 70)
print("FINAL STALENESS SIGNAL VERIFICATION")
print("=" * 70)

display(final_check)


# ============================================================
# 13. DECAY RATE ORDER
# ============================================================

rates = (
    final_check[
        "decay_rate"
    ]
    .dropna()
    .tolist()
)

print("\nDecay rates in age order:")
print(rates)


# ============================================================
# 14. FINAL VERDICT
# ============================================================
#
# CONFIRMED:
#     Older content → higher decay
#
# OPPOSITE:
#     Younger content → higher decay
#
# MIXED:
#     No consistent relationship
# ============================================================

if len(rates) < 2:

    verdict = "INSUFFICIENT DATA"

elif all(
    rates[i] < rates[i + 1]
    for i in range(len(rates) - 1)
):

    verdict = "CONFIRMED"

elif all(
    rates[i] > rates[i + 1]
    for i in range(len(rates) - 1)
):

    verdict = "OPPOSITE"

else:

    verdict = "MIXED"


print("\n" + "=" * 70)
print("FINAL VERDICT:", verdict)
print("=" * 70)


# ============================================================
# 15. ORIGINAL DATAFRAME CHECK
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL DATAFRAME CHECK")
print("=" * 70)

print(
    f"Original df_baseline rows    : "
    f"{len(df_baseline):,}"
)

print(
    f"Original df_baseline columns : "
    f"{len(df_baseline.columns)}"
)

print(
    "\n✓ df_baseline was NOT modified."
)

/tmp/ipykernel_673/1240361124.py:174: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[False False False ...  True  True  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_decay_test.loc[


FINAL STALENESS SIGNAL VERIFICATION


,age_bucket,observations,decay_cases,decay_rate,avg_future_impression_change_pct
0,90-180 days,103259,64667,62.626018,727.89
1,181-365 days,588405,274642,46.675674,780.78
2,365+ days,188441,73565,39.038744,391.70



Decay rates in age order:
[62.626018071064024, 46.67567406803138, 39.038744222329534]

FINAL VERDICT: OPPOSITE

ORIGINAL DATAFRAME CHECK
Original df_baseline rows    : 2,871,202
Original df_baseline columns : 19

✓ df_baseline was NOT modified.


**Signal Test # 2**: CTR VS decay

In [11]:
import pandas as pd
import numpy as np

# ================================================================
# SIGNAL TEST #2 — CTR vs FUTURE DECAY
# ================================================================

# Use your current final dataframe
df_test = df_features.copy()

# Make sure month is datetime
df_test["month"] = pd.to_datetime(df_test["month"])

# Sort page history
df_test = df_test.sort_values(
    ["client_hash_id", "content_hash_id", "month"]
).reset_index(drop=True)


# ================================================================
# 1. CREATE FUTURE 3-MONTH IMPRESSION CHANGE
# ================================================================

# Future impression = impression after 3 months
df_test["future_impressions_3m"] = (
    df_test.groupby(["client_hash_id", "content_hash_id"])["gsc_impressions"]
    .shift(-3)
)

# Current impressions
df_test["current_impressions"] = df_test["gsc_impressions"]


# Future percentage change
# Only calculate where current impressions > 0
df_test["future_impression_change_pct"] = np.where(
    df_test["current_impressions"] > 0,
    (
        (
            df_test["future_impressions_3m"]
            - df_test["current_impressions"]
        )
        / df_test["current_impressions"]
    ) * 100,
    np.nan
)


# ================================================================
# 2. KEEP ONLY OBSERVATIONS WITH A REAL 3-MONTH FUTURE
# ================================================================

df_ctr_test = df_test[
    df_test["future_impression_change_pct"].notna()
].copy()


# ================================================================
# 3. CREATE CTR BUCKETS
# ================================================================

# CTR is already available as a derived feature.
# Convert to percentage for easier interpretation.

df_ctr_test["ctr_percent"] = df_ctr_test["ctr"] * 100

def ctr_bucket(ctr):
    if pd.isna(ctr):
        return np.nan
    elif ctr < 0.5:
        return "Very Low (<0.5%)"
    elif ctr < 1.0:
        return "Low (0.5–1%)"
    elif ctr < 2.0:
        return "Medium (1–2%)"
    elif ctr < 5.0:
        return "High (2–5%)"
    else:
        return "Very High (5%+)"

df_ctr_test["ctr_bucket"] = df_ctr_test["ctr_percent"].apply(ctr_bucket)


# ================================================================
# 4. DEFINE FUTURE DECAY
# ================================================================

# Future decay = impressions decrease by 20% or more
df_ctr_test["future_decay"] = (
    df_ctr_test["future_impression_change_pct"] <= -20
)


# ================================================================
# 5. BUCKET SUMMARY
# ================================================================

bucket_order = [
    "Very Low (<0.5%)",
    "Low (0.5–1%)",
    "Medium (1–2%)",
    "High (2–5%)",
    "Very High (5%+)"
]

ctr_results = (
    df_ctr_test
    .groupby("ctr_bucket", observed=True)
    .agg(
        observations=("future_decay", "size"),
        decay_cases=("future_decay", "sum"),
        avg_future_impression_change_pct=(
            "future_impression_change_pct",
            "mean"
        ),
        median_future_impression_change_pct=(
            "future_impression_change_pct",
            "median"
        )
    )
    .reindex(bucket_order)
    .reset_index()
)

ctr_results["decay_rate"] = (
    ctr_results["decay_cases"]
    / ctr_results["observations"]
    * 100
)


# ================================================================
# 6. DISPLAY RESULT
# ================================================================

print("=" * 70)
print("SIGNAL TEST #2 — CTR vs FUTURE DECAY")
print("=" * 70)

print("\nHypothesis:")
print("Lower CTR should be associated with higher future decay risk.")

print("\nDefinition:")
print("Future decay = future 3-month impressions decrease by >= 20%.")

print("\n" + "=" * 70)
print("CTR BUCKET RESULTS")
print("=" * 70)

display(
    ctr_results[
        [
            "ctr_bucket",
            "observations",
            "decay_cases",
            "decay_rate",
            "avg_future_impression_change_pct",
            "median_future_impression_change_pct"
        ]
    ]
)


# ================================================================
# 7. AUTOMATIC VERDICT
# ================================================================

valid = ctr_results.dropna(subset=["decay_rate"])

if len(valid) >= 2:

    lowest_ctr_decay = valid.iloc[0]["decay_rate"]
    highest_ctr_decay = valid.iloc[-1]["decay_rate"]

    # Expected:
    # Low CTR -> high decay
    # High CTR -> low decay

    if highest_ctr_decay < lowest_ctr_decay:
        verdict = "CONFIRMED"

    elif highest_ctr_decay > lowest_ctr_decay:
        verdict = "OPPOSITE"

    else:
        verdict = "MIXED"

    # Check whether decay generally decreases as CTR increases
    decay_values = valid["decay_rate"].values

    increasing = all(
        decay_values[i] <= decay_values[i + 1]
        for i in range(len(decay_values) - 1)
    )

    decreasing = all(
        decay_values[i] >= decay_values[i + 1]
        for i in range(len(decay_values) - 1)
    )

    if decreasing:
        verdict = "CONFIRMED"

    elif increasing:
        verdict = "OPPOSITE"

    else:
        verdict = "MIXED"

else:
    verdict = "FALSE"


# ================================================================
# 8. FINAL RESULT
# ================================================================

print("\n" + "=" * 70)
print("FINAL VERDICT")
print("=" * 70)

print(f"CTR observations tested : {len(df_ctr_test):,}")
print(f"VERDICT                  : {verdict}")

if verdict == "CONFIRMED":
    print(
        "Lower CTR is associated with higher future decay."
    )

elif verdict == "OPPOSITE":
    print(
        "Higher CTR is associated with higher future decay."
    )

elif verdict == "MIXED":
    print(
        "CTR shows no consistent monotonic relationship with future decay."
    )

else:
    print(
        "Insufficient usable data to evaluate the CTR signal."
    )

SIGNAL TEST #2 — CTR vs FUTURE DECAY

Hypothesis:
Lower CTR should be associated with higher future decay risk.

Definition:
Future decay = future 3-month impressions decrease by >= 20%.

CTR BUCKET RESULTS


,ctr_bucket,observations,decay_cases,decay_rate,avg_future_impression_change_pct,median_future_impression_change_pct
0,Very Low (<0.5%),751162,370297,49.296557,742.151395,-16.831956
1,Low (0.5–1%),65394,21512,32.895984,168.502048,27.895125
2,Medium (1–2%),34564,9738,28.173822,322.694558,58.948645
3,High (2–5%),16614,5015,30.185386,701.607153,95.961199
4,Very High (5%+),12371,6312,51.022553,1380.945240,-25.000000



FINAL VERDICT
CTR observations tested : 880,105
VERDICT                  : MIXED
CTR shows no consistent monotonic relationship with future decay.


**Experiment:**

In [12]:
# ============================================================
# W04 — IMPRESSION MOMENTUM COVERAGE TEST
# ============================================================
#
# Purpose:
#   Sirf ye check karna hai ke impression_momentum
#   kitni monthly rows par calculate ho sakta hai.
#
# IMPORTANT:
#   df_features ORIGINAL hai.
#   Usko modify nahi karna.
#
# No target
# No scoring
# No ranking
# No imputation
#
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — IMPRESSION MOMENTUM COVERAGE TEST")
print("=" * 80)


# ============================================================
# 1. COPY ONLY
# ============================================================

df_momentum_test = df_features.copy()

print(
    f"\nOriginal rows    : {len(df_features):,}"
)

print(
    f"Original columns : {len(df_features.columns)}"
)


# ============================================================
# 2. MONTH FORMAT + SORT COPY
# ============================================================

df_momentum_test["month"] = pd.to_datetime(
    df_momentum_test["month"],
    errors="coerce"
)

df_momentum_test = (
    df_momentum_test
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 3. PREVIOUS MONTH IMPRESSIONS
# ============================================================

previous_impressions = (
    df_momentum_test
    .groupby(
        "content_hash_id",
        sort=False
    )["gsc_impressions"]
    .shift(1)
)


# ============================================================
# 4. CHECK WHETHER MOMENTUM CAN BE CALCULATED
# ============================================================
#
# Requirements:
#
#   Current impressions available
#   +
#   Previous impressions available
#   +
#   Previous impressions > 0
#
# We DO NOT replace missing momentum with 0.
# ============================================================

current_impressions = (
    df_momentum_test["gsc_impressions"]
)

momentum_valid = (
    current_impressions.notna()
    &
    previous_impressions.notna()
    &
    np.isfinite(current_impressions)
    &
    np.isfinite(previous_impressions)
    &
    (previous_impressions > 0)
)


# ============================================================
# 5. CALCULATE MOMENTUM ONLY WHERE VALID
# ============================================================

df_momentum_test["impression_momentum"] = np.nan

df_momentum_test.loc[
    momentum_valid,
    "impression_momentum"
] = (
    (
        current_impressions.loc[momentum_valid]
        -
        previous_impressions.loc[momentum_valid]
    )
    /
    previous_impressions.loc[momentum_valid]
) * 100


# ============================================================
# 6. COVERAGE NUMBERS
# ============================================================

total_rows = len(df_momentum_test)

scored_rows = int(
    df_momentum_test["impression_momentum"]
    .notna()
    .sum()
)

unscored_rows = (
    total_rows - scored_rows
)

scored_pct = (
    scored_rows
    /
    total_rows
    *
    100
)

unscored_pct = (
    unscored_rows
    /
    total_rows
    *
    100
)


# ============================================================
# 7. RESULT
# ============================================================

print("\n" + "=" * 80)
print("IMPRESSION MOMENTUM COVERAGE")
print("=" * 80)

print(
    f"Total monthly rows : {total_rows:,}"
)

print(
    f"Momentum scored    : {scored_rows:,}"
)

print(
    f"Momentum unscored  : {unscored_rows:,}"
)

print(
    f"Scored percentage  : {scored_pct:.2f}%"
)

print(
    f"Unscored percentage: {unscored_pct:.2f}%"
)


# ============================================================
# 8. WHY UNSCORED?
# ============================================================

first_observation_rows = (
    previous_impressions.isna()
)

zero_previous_rows = (
    previous_impressions.eq(0)
    &
    previous_impressions.notna()
)

other_invalid_rows = (
    ~momentum_valid
    &
    ~first_observation_rows
    &
    ~zero_previous_rows
)

print("\n" + "=" * 80)
print("UNSCORED ROW BREAKDOWN")
print("=" * 80)

print(
    f"No previous observation : "
    f"{int(first_observation_rows.sum()):,}"
)

print(
    f"Previous impressions = 0: "
    f"{int(zero_previous_rows.sum()):,}"
)

print(
    f"Other invalid cases     : "
    f"{int(other_invalid_rows.sum()):,}"
)


# ============================================================
# 9. MOMENTUM RANGE
# ============================================================

valid_momentum = (
    df_momentum_test[
        "impression_momentum"
    ].dropna()
)

print("\n" + "=" * 80)
print("VALID MOMENTUM RANGE")
print("=" * 80)

if len(valid_momentum) > 0:

    print(
        f"Minimum momentum : "
        f"{valid_momentum.min():.2f}%"
    )

    print(
        f"Maximum momentum : "
        f"{valid_momentum.max():.2f}%"
    )

    print(
        f"Median momentum  : "
        f"{valid_momentum.median():.2f}%"
    )


# ============================================================
# 10. FINAL DECISION GUIDANCE
# ============================================================

print("\n" + "=" * 80)
print("DECISION CHECK")
print("=" * 80)

if unscored_pct <= 10:

    print(
        "✓ Unscored percentage is MINOR."
    )

    print(
        "→ Impression Momentum can remain "
        "a candidate for the priority signal."
    )

elif unscored_pct <= 20:

    print(
        "⚠ Unscored percentage is MODERATE."
    )

    print(
        "→ Impression Momentum is usable, "
        "but complete-coverage signals should "
        "also be compared."
    )

else:

    print(
        "✗ Unscored percentage is HIGH."
    )

    print(
        "→ Do NOT use Impression Momentum alone "
        "for the final priority queue."
    )


print("\n" + "=" * 80)
print("ORIGINAL DATAFRAME CHECK")
print("=" * 80)

print(
    f"df_features rows    : {len(df_features):,}"
)

print(
    f"df_features columns : {len(df_features.columns)}"
)

print(
    "\n✓ df_features was NOT modified."
)

W04 — IMPRESSION MOMENTUM COVERAGE TEST

Original rows    : 2,871,202
Original columns : 19

IMPRESSION MOMENTUM COVERAGE
Total monthly rows : 2,871,202
Momentum scored    : 1,329,512
Momentum unscored  : 1,541,690
Scored percentage  : 46.31%
Unscored percentage: 53.69%

UNSCORED ROW BREAKDOWN
No previous observation : 427,292
Previous impressions = 0: 1,114,398
Other invalid cases     : 0

VALID MOMENTUM RANGE
Minimum momentum : -100.00%
Maximum momentum : 843600.00%
Median momentum  : -7.74%

DECISION CHECK
✗ Unscored percentage is HIGH.
→ Do NOT use Impression Momentum alone for the final priority queue.

ORIGINAL DATAFRAME CHECK
df_features rows    : 2,871,202
df_features columns : 19

✓ df_features was NOT modified.


**CODE BLOCK 1 — Signal discovery + best signal selection + 20-column dataframe**

In [13]:
# ============================================================
# W04 — BLOCK 1
# FINAL SIGNAL SELECTION
# ============================================================
#
# FINAL DECISION:
#     gsc_impressions
#
# Reason:
#     100% coverage
#     No previous-month dependency
#     No artificial zero imputation
#     Every monthly observation can be ranked
#
# impression_momentum:
#     REJECTED as primary signal
#     because 53.69% rows were unscored.
#
# IMPORTANT:
#     df_features is NEVER modified.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — FINAL SIGNAL SELECTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. ORIGINAL → CLEAN COPY
# ------------------------------------------------------------

df_signal_features = df_features.copy()

print(
    f"Original rows    : {len(df_features):,}"
)

print(
    f"Original columns : {len(df_features.columns)}"
)

# ------------------------------------------------------------
# 2. FINAL SIGNAL
# ------------------------------------------------------------

selected_signal = "gsc_impressions"

signal_name = "GSC Impressions"

# ------------------------------------------------------------
# 3. VALIDATION
# ------------------------------------------------------------

assert selected_signal in df_signal_features.columns

signal_valid = (
    df_signal_features[selected_signal].notna()
    &
    np.isfinite(
        df_signal_features[selected_signal]
    )
)

print("\n" + "=" * 80)
print("SELECTED SIGNAL")
print("=" * 80)

print(f"Signal name : {signal_name}")
print(f"Column      : {selected_signal}")

print(
    f"Valid rows  : {signal_valid.sum():,}"
)

print(
    f"Missing rows: {(~signal_valid).sum():,}"
)

print(
    f"Coverage    : "
    f"{signal_valid.mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 4. FINAL CLEAN DATAFRAME
# ------------------------------------------------------------
#
# gsc_impressions already exists.
# Therefore:
#
# NO duplicate column.
# NO target.
# NO score.
# NO rank.
#
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SIGNAL FEATURE DATAFRAME")
print("=" * 80)

print(
    f"Rows    : {len(df_signal_features):,}"
)

print(
    f"Columns : {len(df_signal_features.columns)}"
)

assert len(df_signal_features.columns) == 19

print(
    "\n✓ gsc_impressions selected."
)

print(
    "✓ Existing column reused."
)

print(
    "✓ No duplicate signal column created."
)

print(
    "✓ df_features remains untouched."
)

W04 — FINAL SIGNAL SELECTION
Original rows    : 2,871,202
Original columns : 19

SELECTED SIGNAL
Signal name : GSC Impressions
Column      : gsc_impressions
Valid rows  : 2,871,202
Missing rows: 0
Coverage    : 100.00%

FINAL SIGNAL FEATURE DATAFRAME
Rows    : 2,871,202
Columns : 19

✓ gsc_impressions selected.
✓ Existing column reused.
✓ No duplicate signal column created.
✓ df_features remains untouched.


**Continuous monthly score + rank + reason + action**

In [18]:
# ============================================================
# W04 — BLOCK 2
# FINAL MONTHLY IMPRESSION SCORING
# ============================================================
#
# Signal:
#     gsc_impressions
#
# Lower impressions = higher refresh priority
#
# Score:
#     0 to 1
#
# 1.00 = highest priority
# 0.00 = lowest priority
#
# LOG transformation is used because impression values
# can be extremely skewed.
#
# Original df_signal_features is NOT modified.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — FINAL MONTHLY IMPRESSION SCORING")
print("=" * 80)

# ------------------------------------------------------------
# 1. COPY
# ------------------------------------------------------------

df_scoring = df_signal_features.copy()

# ------------------------------------------------------------
# 2. VALID IMPRESSIONS
# ------------------------------------------------------------

impressions = pd.to_numeric(
    df_scoring["gsc_impressions"],
    errors="coerce"
)

valid = (
    impressions.notna()
    &
    np.isfinite(impressions)
    &
    (impressions >= 0)
)

# ------------------------------------------------------------
# 3. LOG TRANSFORMATION
# ------------------------------------------------------------
#
# Example:
#
# 0
# 10
# 100
# 1,000
# 100,000
#
# Raw values are extremely spread out.
#
# log1p compresses the range while preserving ordering.
# ------------------------------------------------------------

log_impressions = pd.Series(
    np.nan,
    index=df_scoring.index,
    dtype="float64"
)

log_impressions.loc[valid] = np.log1p(
    impressions.loc[valid]
)

# ------------------------------------------------------------
# 4. MIN / MAX
# ------------------------------------------------------------

min_log = log_impressions.loc[valid].min()
max_log = log_impressions.loc[valid].max()

# ------------------------------------------------------------
# 5. CONTINUOUS 0–1 PRIORITY SCORE
# ------------------------------------------------------------
#
# Lowest impressions → 1
# Highest impressions → 0
#
# Formula:
#
#     (max - current)
#     ----------------
#       max - min
#
# ------------------------------------------------------------

df_scoring["priority_score"] = np.nan

if max_log > min_log:

    df_scoring.loc[valid, "priority_score"] = (
        (max_log - log_impressions.loc[valid])
        /
        (max_log - min_log)
    )

else:

    # Safety case if all impressions are identical
    df_scoring.loc[valid, "priority_score"] = 0.5


# ------------------------------------------------------------
# 6. FORCE EXACT RANGE
# ------------------------------------------------------------

df_scoring["priority_score"] = (
    df_scoring["priority_score"]
    .clip(0, 1)
)


# ------------------------------------------------------------
# 7. REASON CODE
# ------------------------------------------------------------

df_scoring["reason_code"] = np.where(
    valid,
    "LOW_GSC_IMPRESSIONS",
    "INVALID_GSC_IMPRESSIONS"
)


# ------------------------------------------------------------
# 8. MONTHLY ACTION LABEL
# ------------------------------------------------------------

df_scoring["action_label"] = np.select(
    [
        df_scoring["priority_score"] >= 0.90,

        df_scoring["priority_score"] >= 0.75,

        df_scoring["priority_score"] >= 0.50,

        df_scoring["priority_score"].notna()
    ],
    [
        "URGENT_REFRESH",
        "HIGH_PRIORITY",
        "REVIEW",
        "MONITOR"
    ],
    default="UNSCORED"
)


# ------------------------------------------------------------
# 9. MONTHLY RANK
# ------------------------------------------------------------
#
# Unique rank is created only for queue ordering.
#
# Primary ordering:
#     priority_score DESC
#
# Secondary:
#     gsc_impressions ASC
#
# Final deterministic tie:
#     original row order
# ------------------------------------------------------------

df_scoring["_original_row_order"] = np.arange(
    len(df_scoring)
)

df_scoring["monthly_rank"] = 0

valid_rows = df_scoring.loc[
    valid,
    [
        "priority_score",
        "gsc_impressions",
        "_original_row_order"
    ]
].sort_values(
    [
        "priority_score",
        "gsc_impressions",
        "_original_row_order"
    ],
    ascending=[
        False,
        True,
        True
    ]
)

df_scoring.loc[
    valid_rows.index,
    "monthly_rank"
] = np.arange(
    1,
    len(valid_rows) + 1
)


# ------------------------------------------------------------
# 10. REMOVE TEMPORARY COLUMN
# ------------------------------------------------------------

df_scoring.drop(
    columns=["_original_row_order"],
    inplace=True
)


# ------------------------------------------------------------
# 11. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MONTHLY SCORING VALIDATION")
print("=" * 80)

print(
    f"Total rows       : {len(df_scoring):,}"
)

print(
    f"Scored rows      : {valid.sum():,}"
)

print(
    f"Unscored rows    : {(~valid).sum():,}"
)

print(
    f"Minimum score    : "
    f"{df_scoring['priority_score'].min():.6f}"
)

print(
    f"Maximum score    : "
    f"{df_scoring['priority_score'].max():.6f}"
)

print(
    "\n✓ Monthly score is between 0 and 1."
)

print(
    "✓ Lower impressions receive higher priority."
)

print(
    "✓ Actual zero impressions are preserved."
)

print(
    "✓ Monthly rank is unique."
)

print(
    "✓ Original df_signal_features was NOT modified."
)

W04 — FINAL MONTHLY IMPRESSION SCORING

MONTHLY SCORING VALIDATION
Total rows       : 2,871,202
Scored rows      : 2,871,202
Unscored rows    : 0
Minimum score    : 0.000000
Maximum score    : 1.000000

✓ Monthly score is between 0 and 1.
✓ Lower impressions receive higher priority.
✓ Actual zero impressions are preserved.
✓ Monthly rank is unique.
✓ Original df_signal_features was NOT modified.


**BLOCK 3 — Priority queue + Top 20**

In [19]:
# ============================================================
# W04 — BLOCK 3
# PRIORITY QUEUE — TOP 20
# ============================================================

print("=" * 80)
print("W04 — MONTHLY PRIORITY QUEUE")
print("=" * 80)

priority_queue = (
    df_scoring[
        df_scoring["monthly_rank"] > 0
    ]
    .sort_values(
        "monthly_rank",
        ascending=True
    )
    .head(20)
    .copy()
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(
    priority_queue[
        [
            "monthly_rank",
            "content_hash_id",
            "month",
            "gsc_impressions",
            "priority_score",
            "reason_code",
            "action_label"
        ]
    ]
)

print(
    "\n✓ Priority queue created."
)

print(
    "✓ Top 20 rows selected by monthly rank."
)

print(
    "✓ Rank 1 = highest refresh priority."
)

W04 — MONTHLY PRIORITY QUEUE


,monthly_rank,content_hash_id,month,gsc_impressions,priority_score,reason_code,action_label
18,1,content_00001e488b74b799,2025-11-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
19,2,content_00001e488b74b799,2025-12-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
20,3,content_00001e488b74b799,2026-01-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
21,4,content_00001e488b74b799,2026-02-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
22,5,content_00001e488b74b799,2026-03-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
23,6,content_00001e488b74b799,2026-04-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
24,7,content_00001e488b74b799,2026-05-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
25,8,content_00001e488b74b799,2026-06-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
42,9,content_00008950670cb6b5,2026-02-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH
43,10,content_00008950670cb6b5,2026-03-01,0.0,1.0,LOW_GSC_IMPRESSIONS,URGENT_REFRESH



✓ Priority queue created.
✓ Top 20 rows selected by monthly rank.
✓ Rank 1 = highest refresh priority.


**Block 4 : Page level aggregation**

In [21]:
# ============================================================
# W04 — BLOCK 4
# PAGE-LEVEL AGGREGATION
# ============================================================
#
# FINAL LOGIC:
#
# Monthly rows
#       ↓
# FINAL MONTHLY IMPRESSION SCORE (0–1)
#       ↓
# All monthly scores belonging to one page
#       ↓
# MEAN / AVERAGE
#       ↓
# page_priority_score
#       ↓
# page_rank
#
# 1 PAGE = 1 ROW
#
# Tie-breaking:
#   1. page_priority_score DESC
#   2. max_monthly_score DESC
#   3. min_monthly_impressions ASC
#   4. content_hash_id ASC
#
# Original df_scoring remains untouched.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — PAGE-LEVEL PRIORITY AGGREGATION")
print("=" * 80)


# ============================================================
# 1. COPY ONLY
# ============================================================

df_page = df_scoring.copy()

print("\nWorking on COPY only.")


# ============================================================
# 2. PAGE-LEVEL AGGREGATION
# ============================================================
#
# mean_monthly_score:
#
#     Average of the FINAL MONTHLY IMPRESSION SCORES
#     calculated in Block 2.
#
# Example:
#
# Page A:
# Jan = 0.90
# Feb = 0.70
# Mar = 0.80
# Apr = 0.50
#
# mean_monthly_score =
# (0.90 + 0.70 + 0.80 + 0.50) / 4
# = 0.725
#
# This becomes the page's main priority score.
# ============================================================

df_page_priority = (
    df_page
    .groupby(
        "content_hash_id",
        as_index=False
    )
    .agg(

        # ----------------------------------------------------
        # MAIN PAGE SCORE
        # Average of monthly 0–1 scores
        # ----------------------------------------------------

        mean_monthly_score=(
            "priority_score",
            "mean"
        ),

        # Same value as main score.
        # Kept with explicit name for final ranking.
        page_priority_score=(
            "priority_score",
            "mean"
        ),

        # ----------------------------------------------------
        # Additional information for tie-breaking / analysis
        # ----------------------------------------------------

        max_monthly_score=(
            "priority_score",
            "max"
        ),

        min_monthly_impressions=(
            "gsc_impressions",
            "min"
        ),

        monthly_observations=(
            "priority_score",
            "count"
        ),

        urgent_months=(
            "action_label",
            lambda x: (
                x == "URGENT_REFRESH"
            ).sum()
        ),

        high_priority_months=(
            "action_label",
            lambda x: (
                x == "HIGH_PRIORITY"
            ).sum()
        )
    )
)


# ============================================================
# 3. PAGE REASON CODE
# ============================================================

df_page_priority["reason_code"] = np.select(
    [
        df_page_priority[
            "page_priority_score"
        ] >= 0.90,

        df_page_priority[
            "page_priority_score"
        ] >= 0.75,

        df_page_priority[
            "page_priority_score"
        ] >= 0.50
    ],
    [
        "CONSISTENTLY_LOW_GSC_IMPRESSIONS",
        "HIGH_AVERAGE_GSC_PRIORITY",
        "MODERATE_GSC_PRIORITY"
    ],
    default="LOW_GSC_PRIORITY"
)


# ============================================================
# 4. PAGE ACTION LABEL
# ============================================================

df_page_priority["page_action_label"] = np.select(
    [
        df_page_priority[
            "page_priority_score"
        ] >= 0.90,

        df_page_priority[
            "page_priority_score"
        ] >= 0.75,

        df_page_priority[
            "page_priority_score"
        ] >= 0.50
    ],
    [
        "URGENT_REFRESH",
        "HIGH_PRIORITY",
        "REVIEW"
    ],
    default="MONITOR"
)


# ============================================================
# 5. SORT FOR PRIORITY QUEUE
# ============================================================
#
# Main:
#     Higher page score first
#
# Tie-breaker 1:
#     Higher maximum monthly score
#
# Tie-breaker 2:
#     Lower minimum impressions
#
# Tie-breaker 3:
#     content_hash_id
#
# ============================================================

df_page_priority = (
    df_page_priority
    .sort_values(
        [
            "page_priority_score",
            "max_monthly_score",
            "min_monthly_impressions",
            "content_hash_id"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. UNIQUE PAGE RANK
# ============================================================

df_page_priority["page_rank"] = (
    np.arange(
        1,
        len(df_page_priority) + 1
    )
)


# ============================================================
# 7. TOP 20 PAGES
# ============================================================

top_20_pages = (
    df_page_priority
    .head(20)
    .copy()
)


# ============================================================
# 8. DISPLAY TOP 20
# ============================================================

print("\n" + "=" * 80)
print("TOP 20 PAGE-LEVEL PRIORITY QUEUE")
print("=" * 80)

display(
    top_20_pages[
        [
            "page_rank",
            "content_hash_id",

            # Main score + explicit average
            "mean_monthly_score",
            "page_priority_score",

            # Supporting information
            "max_monthly_score",
            "monthly_observations",
            "urgent_months",
            "high_priority_months",

            # Signal information
            "min_monthly_impressions",

            # Decision
            "reason_code",
            "page_action_label"
        ]
    ]
)


# ============================================================
# 9. VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("PAGE-LEVEL VALIDATION")
print("=" * 80)

print(
    f"Monthly rows       : {len(df_scoring):,}"
)

print(
    f"Unique pages       : "
    f"{df_page_priority['content_hash_id'].nunique():,}"
)

print(
    f"Page rows          : "
    f"{len(df_page_priority):,}"
)

print(
    f"Top 20 pages       : "
    f"{len(top_20_pages):,}"
)

print(
    f"\nMinimum page score : "
    f"{df_page_priority['page_priority_score'].min():.6f}"
)

print(
    f"Maximum page score : "
    f"{df_page_priority['page_priority_score'].max():.6f}"
)


# ============================================================
# 10. CONSISTENCY CHECK
# ============================================================
#
# mean_monthly_score and page_priority_score MUST be identical.
# ============================================================

score_difference = (
    df_page_priority["mean_monthly_score"]
    -
    df_page_priority["page_priority_score"]
).abs().max()

print(
    f"\nMaximum difference between "
    f"mean_monthly_score and page_priority_score: "
    f"{score_difference:.10f}"
)

if score_difference == 0:
    print(
        "✓ Page priority score = mean of monthly scores."
    )
else:
    print(
        "⚠ Score consistency check requires review."
    )


print("\n✓ 1 page = 1 row.")
print("✓ Monthly 0–1 scores were averaged.")
print("✓ Mean monthly score is explicitly shown.")
print("✓ Reason code attached.")
print("✓ Page action label attached.")
print("✓ Unique page rank created.")
print("✓ Original df_scoring was NOT modified.")

W04 — PAGE-LEVEL PRIORITY AGGREGATION

Working on COPY only.

TOP 20 PAGE-LEVEL PRIORITY QUEUE


,page_rank,content_hash_id,mean_monthly_score,page_priority_score,max_monthly_score,monthly_observations,urgent_months,high_priority_months,min_monthly_impressions,reason_code,page_action_label
0,1,content_00001e488b74b799,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
1,2,content_0000d31f3926ea12,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
2,3,content_0000feb69f0f60db,1.0,1.0,1.0,5,5,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
3,4,content_000144f5c780452e,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
4,5,content_00019dcd11121026,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
5,6,content_0001ad17454444d0,1.0,1.0,1.0,3,3,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
6,7,content_000308203900dd8b,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
7,8,content_0003183176e176f2,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
8,9,content_00036470b65bb8a6,1.0,1.0,1.0,9,9,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
9,10,content_0003b1eb2a5a5762,1.0,1.0,1.0,9,9,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH



PAGE-LEVEL VALIDATION
Monthly rows       : 2,871,202
Unique pages       : 427,292
Page rows          : 427,292
Top 20 pages       : 20

Minimum page score : 0.076266
Maximum page score : 1.000000

Maximum difference between mean_monthly_score and page_priority_score: 0.0000000000
✓ Page priority score = mean of monthly scores.

✓ 1 page = 1 row.
✓ Monthly 0–1 scores were averaged.
✓ Mean monthly score is explicitly shown.
✓ Reason code attached.
✓ Page action label attached.
✓ Unique page rank created.
✓ Original df_scoring was NOT modified.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 4. Weak Picks + Leakage Check

### Weak Picks Analysis
> Humne signals ko future 3-month decay target ke against test kiya tha.

| Signal | Result | Decision |
| :--- | :--- | :--- |
| **GSC Impressions** | 100% coverage, INCREASING relationship, effect ≈ 15.09 pp | ✅ Selected baseline |
| **Average Position** | MIXED, effect ≈ 15.92 pp | ⚠️ Weak / Unreliable |
| **Impression Momentum** | Strongest relationship, effect ≈ 24.17 pp, but only 1,329,512 scored rows | ⚠️ Strong but incomplete |

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
# ============================================================
# W04 — BLOCK 5
# EXPORT TOP 20 PAGE PRIORITY QUEUE
# ============================================================

output_path = "/content/top20_page_priority_queue.csv"

top_20_pages[
    [
        "page_rank",
        "content_hash_id",
        "mean_monthly_score",
        "page_priority_score",
        "max_monthly_score",
        "monthly_observations",
        "urgent_months",
        "high_priority_months",
        "min_monthly_impressions",
        "reason_code",
        "page_action_label"
    ]
].to_csv(
    output_path,
    index=False
)

print("=" * 80)
print("TOP 20 PAGE QUEUE CSV")
print("=" * 80)

print(f"\nCSV saved successfully:")
print(output_path)

print("\n✓ 20-page priority queue exported.")

TOP 20 PAGE QUEUE CSV

CSV saved successfully:
/content/top20_page_priority_queue.csv

✓ 20-page priority queue exported.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
# ============================================================
# W04 — BLOCK 4
# PAGE-LEVEL AGGREGATION
# ============================================================
#
# FINAL LOGIC:
#
# Monthly rows
#       ↓
# FINAL MONTHLY IMPRESSION SCORE (0–1)
#       ↓
# All monthly scores belonging to one page
#       ↓
# MEAN / AVERAGE
#       ↓
# page_priority_score
#       ↓
# page_rank
#
# 1 PAGE = 1 ROW
#
# Tie-breaking:
#   1. page_priority_score DESC
#   2. max_monthly_score DESC
#   3. min_monthly_impressions ASC
#   4. content_hash_id ASC
#
# Original df_scoring remains untouched.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("W04 — PAGE-LEVEL PRIORITY AGGREGATION")
print("=" * 80)


# ============================================================
# 1. COPY ONLY
# ============================================================

df_page = df_scoring.copy()

print("\nWorking on COPY only.")


# ============================================================
# 2. PAGE-LEVEL AGGREGATION
# ============================================================
#
# mean_monthly_score:
#
#     Average of the FINAL MONTHLY IMPRESSION SCORES
#     calculated in Block 2.
#
# Example:
#
# Page A:
# Jan = 0.90
# Feb = 0.70
# Mar = 0.80
# Apr = 0.50
#
# mean_monthly_score =
# (0.90 + 0.70 + 0.80 + 0.50) / 4
# = 0.725
#
# This becomes the page's main priority score.
# ============================================================

df_page_priority = (
    df_page
    .groupby(
        "content_hash_id",
        as_index=False
    )
    .agg(

        # ----------------------------------------------------
        # MAIN PAGE SCORE
        # Average of monthly 0–1 scores
        # ----------------------------------------------------

        mean_monthly_score=(
            "priority_score",
            "mean"
        ),

        # Same value as main score.
        # Kept with explicit name for final ranking.
        page_priority_score=(
            "priority_score",
            "mean"
        ),

        # ----------------------------------------------------
        # Additional information for tie-breaking / analysis
        # ----------------------------------------------------

        max_monthly_score=(
            "priority_score",
            "max"
        ),

        min_monthly_impressions=(
            "gsc_impressions",
            "min"
        ),

        monthly_observations=(
            "priority_score",
            "count"
        ),

        urgent_months=(
            "action_label",
            lambda x: (
                x == "URGENT_REFRESH"
            ).sum()
        ),

        high_priority_months=(
            "action_label",
            lambda x: (
                x == "HIGH_PRIORITY"
            ).sum()
        )
    )
)


# ============================================================
# 3. PAGE REASON CODE
# ============================================================

df_page_priority["reason_code"] = np.select(
    [
        df_page_priority[
            "page_priority_score"
        ] >= 0.90,

        df_page_priority[
            "page_priority_score"
        ] >= 0.75,

        df_page_priority[
            "page_priority_score"
        ] >= 0.50
    ],
    [
        "CONSISTENTLY_LOW_GSC_IMPRESSIONS",
        "HIGH_AVERAGE_GSC_PRIORITY",
        "MODERATE_GSC_PRIORITY"
    ],
    default="LOW_GSC_PRIORITY"
)


# ============================================================
# 4. PAGE ACTION LABEL
# ============================================================

df_page_priority["page_action_label"] = np.select(
    [
        df_page_priority[
            "page_priority_score"
        ] >= 0.90,

        df_page_priority[
            "page_priority_score"
        ] >= 0.75,

        df_page_priority[
            "page_priority_score"
        ] >= 0.50
    ],
    [
        "URGENT_REFRESH",
        "HIGH_PRIORITY",
        "REVIEW"
    ],
    default="MONITOR"
)


# ============================================================
# 5. SORT FOR PRIORITY QUEUE
# ============================================================
#
# Main:
#     Higher page score first
#
# Tie-breaker 1:
#     Higher maximum monthly score
#
# Tie-breaker 2:
#     Lower minimum impressions
#
# Tie-breaker 3:
#     content_hash_id
#
# ============================================================

df_page_priority = (
    df_page_priority
    .sort_values(
        [
            "page_priority_score",
            "max_monthly_score",
            "min_monthly_impressions",
            "content_hash_id"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. UNIQUE PAGE RANK
# ============================================================

df_page_priority["page_rank"] = (
    np.arange(
        1,
        len(df_page_priority) + 1
    )
)


# ============================================================
# 7. TOP 20 PAGES
# ============================================================

top_20_pages = (
    df_page_priority
    .head(20)
    .copy()
)


# ============================================================
# 8. DISPLAY TOP 20
# ============================================================

print("\n" + "=" * 80)
print("TOP 20 PAGE-LEVEL PRIORITY QUEUE")
print("=" * 80)

display(
    top_20_pages[
        [
            "page_rank",
            "content_hash_id",

            # Main score + explicit average
            "mean_monthly_score",
            "page_priority_score",

            # Supporting information
            "max_monthly_score",
            "monthly_observations",
            "urgent_months",
            "high_priority_months",

            # Signal information
            "min_monthly_impressions",

            # Decision
            "reason_code",
            "page_action_label"
        ]
    ]
)


# ============================================================
# 9. VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("PAGE-LEVEL VALIDATION")
print("=" * 80)

print(
    f"Monthly rows       : {len(df_scoring):,}"
)

print(
    f"Unique pages       : "
    f"{df_page_priority['content_hash_id'].nunique():,}"
)

print(
    f"Page rows          : "
    f"{len(df_page_priority):,}"
)

print(
    f"Top 20 pages       : "
    f"{len(top_20_pages):,}"
)

print(
    f"\nMinimum page score : "
    f"{df_page_priority['page_priority_score'].min():.6f}"
)

print(
    f"Maximum page score : "
    f"{df_page_priority['page_priority_score'].max():.6f}"
)


# ============================================================
# 10. CONSISTENCY CHECK
# ============================================================
#
# mean_monthly_score and page_priority_score MUST be identical.
# ============================================================

score_difference = (
    df_page_priority["mean_monthly_score"]
    -
    df_page_priority["page_priority_score"]
).abs().max()

print(
    f"\nMaximum difference between "
    f"mean_monthly_score and page_priority_score: "
    f"{score_difference:.10f}"
)

if score_difference == 0:
    print(
        "✓ Page priority score = mean of monthly scores."
    )
else:
    print(
        "⚠ Score consistency check requires review."
    )


print("\n✓ 1 page = 1 row.")
print("✓ Monthly 0–1 scores were averaged.")
print("✓ Mean monthly score is explicitly shown.")
print("✓ Reason code attached.")
print("✓ Page action label attached.")
print("✓ Unique page rank created.")
print("✓ Original df_scoring was NOT modified.")

W04 — PAGE-LEVEL PRIORITY AGGREGATION

Working on COPY only.

TOP 20 PAGE-LEVEL PRIORITY QUEUE


,page_rank,content_hash_id,mean_monthly_score,page_priority_score,max_monthly_score,monthly_observations,urgent_months,high_priority_months,min_monthly_impressions,reason_code,page_action_label
0,1,content_00001e488b74b799,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
1,2,content_0000d31f3926ea12,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
2,3,content_0000feb69f0f60db,1.0,1.0,1.0,5,5,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
3,4,content_000144f5c780452e,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
4,5,content_00019dcd11121026,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
5,6,content_0001ad17454444d0,1.0,1.0,1.0,3,3,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
6,7,content_000308203900dd8b,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
7,8,content_0003183176e176f2,1.0,1.0,1.0,8,8,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
8,9,content_00036470b65bb8a6,1.0,1.0,1.0,9,9,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH
9,10,content_0003b1eb2a5a5762,1.0,1.0,1.0,9,9,0,0.0,CONSISTENTLY_LOW_GSC_IMPRESSIONS,URGENT_REFRESH



PAGE-LEVEL VALIDATION
Monthly rows       : 2,871,202
Unique pages       : 427,292
Page rows          : 427,292
Top 20 pages       : 20

Minimum page score : 0.076266
Maximum page score : 1.000000

Maximum difference between mean_monthly_score and page_priority_score: 0.0000000000
✓ Page priority score = mean of monthly scores.

✓ 1 page = 1 row.
✓ Monthly 0–1 scores were averaged.
✓ Mean monthly score is explicitly shown.
✓ Reason code attached.
✓ Page action label attached.
✓ Unique page rank created.
✓ Original df_scoring was NOT modified.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Rule — Plain Words

The final priority rule is based on GSC Impressions (`gsc_impressions`).

For every page-month, we look at its GSC impression value and convert it into a monthly priority score between 0 and 1. Lower GSC impressions indicate greater potential need for attention, so a lower-impression observation receives a higher priority score.

### Reason Codes
The rule can attach a reason code to explain why a page received its priority.

#### 1. LOW_GSC_IMPRESSIONS
The page has relatively low GSC impressions, resulting in a higher priority score.

**Meaning:**
* The page currently receives relatively low search impressions and should receive attention.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.